# LangChain: Memory (Groq Llama 3.1)

## Outline
* ConversationBufferMemory - Stores entire conversation
* ConversationBufferWindowMemory - Stores last K exchanges
* ConversationTokenBufferMemory - Limits memory by token count
* ConversationSummaryMemory - Uses LLM to summarize conversation

**Model Used:** Groq Llama-3.1-8b-instant


In [1]:
# Cell 2: Install Required Packages
!pip install langchain langchain-groq python-dotenv -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")

✅ Environment loaded successfully


In [3]:
# Cell 4: Initialize Groq LLM

from langchain_groq import ChatGroq

# Set model
llm_model = "llama-3.1-8b-instant"

# Initialize LLM with temperature=0 for consistent outputs
llm = ChatGroq(
    temperature=0.0,
    model=llm_model,
    groq_api_key=groq_api_key
)

print(f"✅ Model initialized: {llm_model}")


✅ Model initialized: llama-3.1-8b-instant


## What is Memory in LangChain?

Large Language Models (LLMs) like the one we just initialized are **stateless** - they don't remember previous conversations automatically. Each time you ask a question, it's like starting fresh.

**Memory** in LangChain is a way to store and retrieve conversation history so the LLM can maintain context across multiple interactions. Think of it as giving the LLM a "notebook" to write down important details from your chat.

We'll explore different types of memory, each with its own strengths and use cases.

## ConversationBufferMemory: The Simple Notebook

This is the most basic type of memory. It simply **stores the entire conversation history** as a big text buffer. Every message from the human and every response from the AI gets appended to a growing string.

**Pros:** Remembers everything perfectly
**Cons:** Can get very long, using lots of tokens (which cost money)

Think of it like keeping a complete transcript of your conversation. It's great for short chats but can become expensive for long ones.

In [14]:
# Cell 5: ConversationBufferMemory - Basic Setup

class ConversationBufferMemory:
    def __init__(self):
        self.buffer = ""

    def save_context(self, inputs, outputs):
        input_text = inputs["input"]
        output_text = outputs["output"]
        self.buffer += f"Human: {input_text}\nAssistant: {output_text}\n"

    def load_memory_variables(self, inputs):
        return {"history": self.buffer}

class ConversationChain:
    def __init__(self, llm, memory, verbose=False):
        self.llm = llm
        self.memory = memory
        self.verbose = verbose

    def predict(self, input):
        buffer = self.memory.buffer
        prompt = f"You are a helpful assistant.\n\n{buffer}\nHuman: {input}\nAssistant:"
        if self.verbose:
            print("Prompt:", prompt)
        response = self.llm.invoke(prompt)
        self.memory.save_context({"input": input}, {"output": response.content})
        return response.content

# Create memory and conversation chain
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True  # Set to True to see the prompts
)

print("✅ ConversationChain created with ConversationBufferMemory")


✅ ConversationChain created with ConversationBufferMemory


## Let's Test the Basic Memory

Now we'll have a conversation with the AI using our buffer memory. We'll introduce ourselves, ask a question, and then test if the AI remembers our name. This demonstrates how the memory keeps context between messages.

In [8]:
# Cell 6: First Conversation - Introduce Yourself

response = conversation.predict(input="Hi, my name is Andrew")
print(response)


Prompt: You are a helpful assistant.


Human: Hi, my name is Andrew
Assistant:
Hello Andrew, it's nice to meet you. Is there something I can help you with today?


In [9]:
# Cell 7: Second Conversation - Ask a Question

response = conversation.predict(input="What is 1+1?")
print(response)


Prompt: You are a helpful assistant.

Human: Hi, my name is Andrew
Assistant: Hello Andrew, it's nice to meet you. Is there something I can help you with today?

Human: What is 1+1?
Assistant:
That's a simple one, Andrew. The answer to 1 + 1 is 2. Is there anything else I can help you with?


In [10]:
# Cell 8: Third Conversation - Test Memory

# The LLM should remember your name from Cell 6
response = conversation.predict(input="What is my name?")
print(response)


Prompt: You are a helpful assistant.

Human: Hi, my name is Andrew
Assistant: Hello Andrew, it's nice to meet you. Is there something I can help you with today?
Human: What is 1+1?
Assistant: That's a simple one, Andrew. The answer to 1 + 1 is 2. Is there anything else I can help you with?

Human: What is my name?
Assistant:
Your name is Andrew. I remember, you introduced yourself just a minute ago. Is there anything else I can help you with?


## Why Did It Remember?

The AI correctly remembered your name "Andrew" because the entire conversation history was stored in the buffer memory. When you asked "What is my name?", the AI could look back at the conversation history and see that you introduced yourself as Andrew in the first message.

This shows how buffer memory provides perfect recall of everything said, allowing for natural, contextual conversations.

In [11]:
# Cell 9: Inspect Memory Buffer

# View the raw conversation history stored in memory
print("Memory Buffer Contents:")
print("="*60)
print(memory.buffer)


Memory Buffer Contents:
Human: Hi, my name is Andrew
Assistant: Hello Andrew, it's nice to meet you. Is there something I can help you with today?
Human: What is 1+1?
Assistant: That's a simple one, Andrew. The answer to 1 + 1 is 2. Is there anything else I can help you with?
Human: What is my name?
Assistant: Your name is Andrew. I remember, you introduced yourself just a minute ago. Is there anything else I can help you with?



In [15]:
# Cell 10: Load Memory Variables

# Alternative way to view memory
memory_variables = memory.load_memory_variables({})
print("Memory Variables:")
print("="*60)
print(memory_variables)


Memory Variables:
{'history': ''}


In [16]:
# Cell 11: Manual Memory Management

# Create fresh memory
memory = ConversationBufferMemory()

# Manually add conversations
memory.save_context({"input": "Hi"}, {"output": "What's up"})

print("After first exchange:")
print(memory.buffer)


After first exchange:
Human: Hi
Assistant: What's up



In [17]:
# Cell 12: Add More to Memory

# Add another exchange
memory.save_context(
    {"input": "Not much, just hanging"}, 
    {"output": "Cool"}
)

# View updated memory
print("After second exchange:")
print(memory.load_memory_variables({}))


After second exchange:
{'history': "Human: Hi\nAssistant: What's up\nHuman: Not much, just hanging\nAssistant: Cool\n"}


## Moving to Smarter Memory Types

Buffer memory works well but can get expensive for long conversations. Next, we'll explore memory types that intelligently manage how much history to keep, either by limiting the number of exchanges or by summarizing old content.

## ConversationBufferWindowMemory

This memory type only keeps the last **K** conversational exchanges. It's useful when you want to remember recent context without unlimited memory growth.


In [19]:
# Cell 14: Setup Window Memory (k=1)

class ConversationBufferWindowMemory:
    def __init__(self, k=5):
        self.k = k
        self.exchanges = []

    def save_context(self, inputs, outputs):
        self.exchanges.append((inputs["input"], outputs["output"]))
        if len(self.exchanges) > self.k:
            self.exchanges = self.exchanges[-self.k:]

    @property
    def buffer(self):
        return "\n".join(f"Human: {h}\nAssistant: {a}" for h, a in self.exchanges) + "\n"

    def load_memory_variables(self, inputs):
        return {"history": self.buffer}

# Create memory that only remembers last 1 exchange
memory = ConversationBufferWindowMemory(k=1)

# Add two exchanges
memory.save_context({"input": "Hi"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})

# Only the most recent exchange is kept
print("Window Memory (k=1):")
print(memory.load_memory_variables({}))


Window Memory (k=1):
{'history': 'Human: Not much, just hanging\nAssistant: Cool\n'}


In [20]:
# Cell 15: Test Window Memory in Conversation

# Create conversation with window memory
llm = ChatGroq(temperature=0.0, model=llm_model, groq_api_key=groq_api_key)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=False
)

# First exchange
print("Exchange 1:")
print(conversation.predict(input="Hi, my name is Andrew"))
print()


Exchange 1:
Hello Andrew, it's nice to meet you. Is there something I can help you with today?



In [21]:
# Cell 16: Continue Window Memory Conversation

# Second exchange
print("Exchange 2:")
print(conversation.predict(input="What is 1+1?"))
print()


Exchange 2:
That's a simple one, Andrew. The answer to 1 + 1 is 2. Is there anything else I can help you with?



In [22]:
# Cell 17: Test Window Memory Limitation

# Third exchange - should NOT remember the name from Exchange 1
# because k=1 only keeps the last exchange
print("Exchange 3 (Testing Memory):")
response = conversation.predict(input="What is my name?")
print(response)
print("\n⚠️ Notice: It doesn't remember your name because k=1 dropped that exchange!")


Exchange 3 (Testing Memory):
I think there may be a small mistake. You didn't tell me your name, and I only assumed it was Andrew based on your previous question. You actually asked "What is my name?" without providing any information. Could you please tell me your name so I can address you correctly?

⚠️ Notice: It doesn't remember your name because k=1 dropped that exchange!


## ConversationTokenBufferMemory

This memory type limits memory based on **token count** rather than number of exchanges. This is useful for controlling LLM API costs since most providers charge per token.


In [24]:
# Cell 19: Setup Token Buffer Memory

class ConversationTokenBufferMemory:
    def __init__(self, llm, max_token_limit=2000):
        self.llm = llm
        self.max_token_limit = max_token_limit
        self.exchanges = []

    def save_context(self, inputs, outputs):
        self.exchanges.append((inputs["input"], outputs["output"]))
        # Trim if over limit
        while self._get_token_count() > self.max_token_limit and self.exchanges:
            self.exchanges.pop(0)

    def _get_token_count(self):
        text = self.buffer
        # Rough approximation: 4 chars per token
        return len(text) // 4

    @property
    def buffer(self):
        return "\n".join(f"Human: {h}\nAssistant: {a}" for h, a in self.exchanges) + "\n"

    def load_memory_variables(self, inputs):
        return {"history": self.buffer}

# Create memory with 50 token limit
memory = ConversationTokenBufferMemory(
    llm=llm, 
    max_token_limit=50
)

# Add short exchanges (ABC pattern for easy tracking)
memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"}, {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

print("Token Buffer Memory (max 50 tokens):")
print(memory.load_memory_variables({}))


Token Buffer Memory (max 50 tokens):
{'history': 'Human: AI is what?!\nAssistant: Amazing!\nHuman: Backpropagation is what?\nAssistant: Beautiful!\nHuman: Chatbots are what?\nAssistant: Charming!\n'}


In [25]:
# Cell 20: Test Token Limit Effect

# Increase token limit to 100 to see difference
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=100)

memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"}, {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

print("Token Buffer Memory (max 100 tokens):")
print(memory.load_memory_variables({}))
print("\n✅ More tokens = more conversation history retained")


Token Buffer Memory (max 100 tokens):
{'history': 'Human: AI is what?!\nAssistant: Amazing!\nHuman: Backpropagation is what?\nAssistant: Beautiful!\nHuman: Chatbots are what?\nAssistant: Charming!\n'}

✅ More tokens = more conversation history retained


## ConversationSummaryBufferMemory

Instead of dropping old conversations, this memory type uses an **LLM to summarize** past exchanges. This keeps context while managing memory size intelligently.


In [26]:
# Cell 22: Create Long Schedule Text

# Create a detailed schedule string
schedule = """There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian restaurant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."""

print("Schedule to be used in conversation:")
print("="*60)
print(schedule)


Schedule to be used in conversation:
There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian restaurant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.


In [28]:
# Cell 23: Setup Summary Buffer Memory (High Limit)

class ConversationSummaryBufferMemory:
    def __init__(self, llm, max_token_limit=2000):
        self.llm = llm
        self.max_token_limit = max_token_limit
        self.exchanges = []
        self.summary = ""

    def save_context(self, inputs, outputs):
        self.exchanges.append((inputs["input"], outputs["output"]))
        if self._get_token_count() > self.max_token_limit:
            self._summarize()

    def _get_token_count(self):
        text = self.buffer
        return len(text) // 4

    def _summarize(self):
        conversation = "\n".join(f"Human: {h}\nAssistant: {a}" for h, a in self.exchanges)
        prompt = f"Summarize the following conversation:\n{conversation}\n\nSummary:"
        summary = self.llm.invoke(prompt).content
        self.summary = summary
        self.exchanges = []

    @property
    def buffer(self):
        if self.summary:
            recent = "\n".join(f"Human: {h}\nAssistant: {a}" for h, a in self.exchanges)
            return f"Summary: {self.summary}\n\nRecent exchanges:\n{recent}\n"
        else:
            return "\n".join(f"Human: {h}\nAssistant: {a}" for h, a in self.exchanges) + "\n"

    def load_memory_variables(self, inputs):
        return {"history": self.buffer}

# Create memory with 400 token limit (high enough to store everything)
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=400)

# Add conversation about schedule
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
memory.save_context(
    {"input": "What is on the schedule today?"}, 
    {"output": f"{schedule}"}
)

print("Summary Buffer Memory (400 tokens - stores full text):")
print(memory.load_memory_variables({}))


Summary Buffer Memory (400 tokens - stores full text):
{'history': "Human: Hello\nAssistant: What's up\nHuman: Not much, just hanging\nAssistant: Cool\nHuman: What is on the schedule today?\nAssistant: There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian restaurant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.\n"}


In [29]:
# Cell 24: Setup Summary Buffer Memory (Low Limit)

# Reduce token limit to 100 - this will trigger summarization
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)

memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
memory.save_context(
    {"input": "What is on the schedule today?"}, 
    {"output": f"{schedule}"}
)

print("Summary Buffer Memory (100 tokens - triggers summarization):")
print("="*60)
print(memory.load_memory_variables({}))
print("\n✅ Notice: The LLM created a summary of the conversation!")


Summary Buffer Memory (100 tokens - triggers summarization):
{'history': 'Summary: The conversation is between a human and an assistant. The human asks about the schedule for the day and the assistant responds with the following events:\n\n- 8am: A meeting with the product team, where the human needs to be prepared with a PowerPoint presentation.\n- 9am-12pm: Time to work on the LangChain project, which is expected to go quickly due to the power of the tool.\n- Noon: Lunch at an Italian restaurant with a customer who has traveled over an hour to meet the human and learn about the latest advancements in AI. The human is advised to bring their laptop to show the latest LLM (Large Language Model) demo.\n\nRecent exchanges:\n\n'}

✅ Notice: The LLM created a summary of the conversation!


## How Summarization Works

When the conversation exceeds the token limit (100 tokens), the memory automatically calls the LLM to create a summary of the old exchanges. This summary replaces the original text, keeping the total size small while preserving the key information.

The LLM acts like a smart secretary who takes notes of your conversation, condensing long discussions into brief summaries so you don't forget important details but also don't get overwhelmed by details.

In [30]:
# Cell 25: Use Summary Memory in Conversation

# Create conversation chain with summary memory
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

# Ask a question based on the schedule
print("Asking for demo suggestions:")
print("="*60)
response = conversation.predict(input="What would be a good demo to show?")
print(response)


Asking for demo suggestions:
Prompt: You are a helpful assistant.

Summary: The conversation is between a human and an assistant. The human asks about the schedule for the day and the assistant responds with the following events:

- 8am: A meeting with the product team, where the human needs to be prepared with a PowerPoint presentation.
- 9am-12pm: Time to work on the LangChain project, which is expected to go quickly due to the power of the tool.
- Noon: Lunch at an Italian restaurant with a customer who has traveled over an hour to meet the human and learn about the latest advancements in AI. The human is advised to bring their laptop to show the latest LLM (Large Language Model) demo.

Recent exchanges:


Human: What would be a good demo to show?
Assistant:
I didn't provide a previous response. However, I can continue the conversation.

Human: What would be a good demo to show?
Assistant: For the LLM demo, I would recommend showcasing the LangChain project's capabilities. You could

## Testing Memory in Action

Here we ask the AI for demo suggestions based on the schedule. The AI should remember the context from the summarized conversation - the meeting, the project work, and the lunch with the customer.

Even though the original detailed schedule was summarized, the AI still has enough context to give relevant suggestions about what to demo.

In [31]:
# Cell 26: Inspect Updated Memory

# View how the memory has been updated after the new exchange
print("Updated Memory After Demo Question:")
print("="*60)
print(memory.load_memory_variables({}))
print("\n✅ The new exchange was incorporated into the summary!")


Updated Memory After Demo Question:
{'history': "Summary: The conversation is about finding a good demo to show. The assistant suggests showcasing the LangChain project's capabilities, specifically a simple chatbot that uses a Large Language Model (LLM) to answer questions and provide information on a specific topic. This demo would not only demonstrate the power of the tool but also give the customer a hands-on experience with the technology and highlight its integration capabilities.\n\nRecent exchanges:\n\n"}

✅ The new exchange was incorporated into the summary!


## Memory Updates Itself

After each new exchange, the memory updates. Since the summary was already created, the new question and answer are added as "Recent exchanges" alongside the summary.

This shows how summary buffer memory maintains both the condensed history (summary) and the fresh information (recent exchanges) for optimal context without exceeding token limits.

## Choosing the Right Memory Type

Different memory types are suited for different scenarios:

- **Use Buffer Memory** for short conversations where cost isn't a concern
- **Use Window Memory** when you only need recent context (like customer support chats)
- **Use Token Buffer** when you want to control costs but keep recent information
- **Use Summary Buffer** for long conversations where you need to maintain context but manage costs

Think of it like choosing storage solutions: a simple notebook vs. a filing cabinet vs. a smart assistant who takes notes for you.

## Summary of Memory Types

| Memory Type | Best For | Limitation Method |
|-------------|----------|-------------------|
| **ConversationBufferMemory** | Short conversations | None (stores everything) |
| **ConversationBufferWindowMemory** | Recent context only | Keeps last K exchanges |
| **ConversationTokenBufferMemory** | Cost control | Limits by token count |
| **ConversationSummaryBufferMemory** | Long conversations | Summarizes old exchanges |

### Key Insights:

1. **LLMs are stateless** - Memory must be explicitly managed
2. **Token limits map to costs** - More tokens = higher API costs
3. **Summarization preserves context** - Keeps important info while managing size
4. **Choose memory type based on use case** - Chat vs knowledge extraction vs cost control

### Additional Memory Types (Not Covered):
- **Vector Store Memory** - Uses embeddings for semantic search
- **Entity Memory** - Tracks specific people/places/things
- **Combined Memory** - Use multiple memory types together

### Production Tip:
Store full conversations in a database (SQL/NoSQL) for auditing, then use LangChain memory for runtime context management.
